In [1]:
"""
benchmark_deep_gnn_cora.py

Consolidated Benchmark: Over-Smoothing Mitigation in Deep Graph Convolutional Networks
Dataset: Cora (Planetoid Split: 140 Train / 500 Validation / 1000 Blind Test)
Evaluation: Depths L in [2, 4, 8, 16, 32, 64] across 5 independent seeds.

Architectures Evaluated:
  - Vanilla GCN: Standard Graph Convolution (baseline)
  - LayerNorm GCN: Node-wise feature normalization (baseline)
  - PairNorm GCN: Total variance regularization (Zhao & Akoglu, ICLR 2020)
  - Graph-Conical: Spherical L2 channel normalization
  - Graph-Equatorial: Zero-mean node projection on S^{N-2} with Weight Standardization

Audited Metrics:
  - Blind Test Accuracy (Mean +/- Std)
  - Active Pairwise Cosine Distance (Diversity)
  - Normalized Dirichlet Energy (Smoothness)
  - Dead Node Percentage (% nodes with L2 norm < 1e-4)
"""

import os
import time
import urllib.request
from typing import Dict, List, Tuple

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

# -----------------------------------------------------------------------------
# 0. Global Setup and Hardware Configuration
# -----------------------------------------------------------------------------
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SEEDS: List[int] = [42, 1337, 2026, 777, 999]
DEPTHS: List[int] = [2, 4, 8, 16, 32, 64]
EPS: float = 1e-7

torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True


# -----------------------------------------------------------------------------
# 1. Resilient Cora Dataset Loader (Planetoid Canonical Split)
# -----------------------------------------------------------------------------
def download_file_with_mirrors(mirrors: List[str], destination_path: str) -> bool:
    """Attempts to download a file from a sequence of mirror URLs."""
    headers = {"User-Agent": "Mozilla/5.0"}
    for url in mirrors:
        try:
            req = urllib.request.Request(url, headers=headers)
            with urllib.request.urlopen(req, timeout=20) as resp, open(destination_path, "wb") as f:
                f.write(resp.read())
            if os.path.exists(destination_path) and os.path.getsize(destination_path) > 1000:
                return True
        except Exception:
            continue
    return False


def load_cora_dataset() -> Tuple[
    torch.Tensor, torch.Tensor, torch.Tensor,
    torch.Tensor, torch.Tensor, torch.Tensor,
    int, int
]:
    """
    Downloads and parses the Cora citation network.
    Constructs the symmetrically normalized adjacency matrix:
        A_hat = D^{-1/2} (A + I) D^{-1/2}
    Applies canonical Planetoid splits: 20 nodes per class for training.
    """
    data_dir = "/tmp/cora_dataset"
    os.makedirs(data_dir, exist_ok=True)
    content_path = os.path.join(data_dir, "cora.content")
    cites_path = os.path.join(data_dir, "cora.cites")

    content_mirrors = [
        "https://raw.githubusercontent.com/Diego999/pyGAT/master/data/cora/cora.content",
        "https://raw.githubusercontent.com/ChandlerL/GCN/master/data/cora/cora.content",
        "https://raw.githubusercontent.com/daunnn/Graph-Neural-Networks/master/cora/cora.content",
    ]
    cites_mirrors = [
        "https://raw.githubusercontent.com/Diego999/pyGAT/master/data/cora/cora.cites",
        "https://raw.githubusercontent.com/ChandlerL/GCN/master/data/cora/cora.cites",
        "https://raw.githubusercontent.com/daunnn/Graph-Neural-Networks/master/cora/cora.cites",
    ]

    if not os.path.exists(content_path) or os.path.getsize(content_path) < 1000:
        if not download_file_with_mirrors(content_mirrors, content_path):
            raise RuntimeError("Failed to retrieve cora.content from all mirrors.")

    if not os.path.exists(cites_path) or os.path.getsize(cites_path) < 1000:
        if not download_file_with_mirrors(cites_mirrors, cites_path):
            raise RuntimeError("Failed to retrieve cora.cites from all mirrors.")

    raw_content = np.genfromtxt(content_path, dtype=str)
    node_ids = raw_content[:, 0]
    features = np.array(raw_content[:, 1:-1], dtype=np.float32)
    labels_raw = raw_content[:, -1]

    unique_labels = {label: idx for idx, label in enumerate(np.unique(labels_raw))}
    labels = np.array([unique_labels[lab] for lab in labels_raw], dtype=np.int64)

    id_to_idx = {nid: idx for idx, nid in enumerate(node_ids)}
    num_nodes = len(node_ids)
    num_features = features.shape[1]
    num_classes = len(unique_labels)

    raw_cites = np.genfromtxt(cites_path, dtype=str)
    adj = np.zeros((num_nodes, num_nodes), dtype=np.float32)
    for source, target in raw_cites:
        if source in id_to_idx and target in id_to_idx:
            u, v = id_to_idx[source], id_to_idx[target]
            adj[u, v] = 1.0
            adj[v, u] = 1.0

    # Symmetric normalized adjacency with self-loops
    adj_tilde = adj + np.eye(num_nodes, dtype=np.float32)
    degrees = np.sum(adj_tilde, axis=1)
    deg_inv_sqrt = np.power(degrees, -0.5)
    deg_inv_sqrt[np.isinf(deg_inv_sqrt)] = 0.0
    adj_hat = deg_inv_sqrt[:, None] * adj_tilde * deg_inv_sqrt[None, :]

    # Canonical Planetoid partitioning
    train_mask = np.zeros(num_nodes, dtype=bool)
    for c in range(num_classes):
        class_indices = np.where(labels == c)[0]
        train_mask[class_indices[:20]] = True

    remaining = np.where(~train_mask)[0]
    val_mask = np.zeros(num_nodes, dtype=bool)
    val_mask[remaining[:500]] = True
    test_mask = np.zeros(num_nodes, dtype=bool)
    test_mask[remaining[500:1500]] = True

    return (
        torch.tensor(features, dtype=torch.float32).to(DEVICE),
        torch.tensor(adj_hat, dtype=torch.float32).to(DEVICE),
        torch.tensor(labels, dtype=torch.long).to(DEVICE),
        torch.tensor(train_mask).to(DEVICE),
        torch.tensor(val_mask).to(DEVICE),
        torch.tensor(test_mask).to(DEVICE),
        num_features,
        num_classes,
    )


# -----------------------------------------------------------------------------
# 2. Structural & Health Metrics
# -----------------------------------------------------------------------------
def compute_graph_metrics_safe(
    features: torch.Tensor,
    adj_hat: torch.Tensor,
    eps: float = 1e-7
) -> Tuple[float, float, float, float]:
    """
    Audits graph representations for signs of over-smoothing or collapse:
      1. Pairwise Cosine Distance: 1 - mean(cos(h_i, h_j)) over active nodes.
      2. Normalized Dirichlet Energy: tr(H^T (I - A_hat) H) / tr(H^T H).
      3. Stable Rank: ||H||_F^2 / ||H||_2^2 of centered latents.
      4. Dead Nodes (%): Proportion of nodes with L2 norm < 1e-4.
    """
    with torch.no_grad():
        norms = torch.norm(features, p=2, dim=-1)
        dead_mask = norms < 1e-4
        dead_ratio = float(dead_mask.float().mean().item() * 100.0)

        # If >80% of nodes collapsed to zero, diversity is null
        if dead_ratio > 80.0:
            return 0.0, 0.0, 0.0, dead_ratio

        active_indices = ~dead_mask
        h_active = features[active_indices]
        h_norm = F.normalize(h_active, p=2, dim=-1, eps=eps)
        pairwise_dist = float((1.0 - torch.matmul(h_norm, h_norm.t())).mean().item())

        h_fro = features / (torch.norm(features, p="fro") + eps)
        laplacian_h = h_fro - torch.matmul(adj_hat, h_fro)
        dirichlet_energy = float(torch.trace(torch.matmul(h_fro.t(), laplacian_h)).item())

        h_centered = (features - features.mean(dim=0, keepdim=True)).cpu()
        try:
            _, singular_vals, _ = torch.linalg.svd(h_centered, full_matrices=False)
            stable_rank = float(
                (torch.sum(singular_vals ** 2) / (torch.max(singular_vals) ** 2 + eps)).item()
            )
        except Exception:
            stable_rank = 0.0

    return pairwise_dist, dirichlet_energy, stable_rank, dead_ratio


# -----------------------------------------------------------------------------
# 3. Model Architecture Factory
# -----------------------------------------------------------------------------
class DeepGraphModel(nn.Module):
    """
    Multi-layer Graph Convolutional Network without residual bypasses.
    Serves as a depth stress-test for message-passing representations.
    """
    def __init__(
        self,
        in_dim: int,
        hidden_dim: int,
        num_classes: int,
        num_layers: int,
        norm_type: str = "equatorial"
    ) -> None:
        super().__init__()
        self.num_layers = num_layers
        self.norm_type = norm_type
        self.input_proj = nn.Linear(in_dim, hidden_dim)

        self.weights = nn.ParameterList([
            nn.Parameter(torch.empty(hidden_dim, hidden_dim))
            for _ in range(num_layers)
        ])
        for w in self.weights:
            nn.init.orthogonal_(w)

        if norm_type == "layernorm":
            self.norms = nn.ModuleList([nn.LayerNorm(hidden_dim) for _ in range(num_layers)])

        self.head = nn.Linear(hidden_dim, num_classes)

    def extract_features(self, x: torch.Tensor, adj_hat: torch.Tensor) -> torch.Tensor:
        """
        Forward message passing through all propagation layers.
        Shape: [N, D_in] -> [N, D_hidden]
        """
        h = self.input_proj(x)
        d_scale = float(h.size(-1) ** 0.5)

        for l in range(self.num_layers):
            w = self.weights[l]
            ax = torch.matmul(adj_hat, h)

            if self.norm_type == "vanilla":
                h = F.relu(torch.matmul(ax, w))

            elif self.norm_type == "layernorm":
                h = torch.matmul(ax, w)
                h = F.relu(self.norms[l](h))

            elif self.norm_type == "pairnorm":
                h = torch.matmul(ax, w)
                h_cent = h - h.mean(dim=0, keepdim=True)
                var = torch.mean(torch.norm(h_cent, p=2, dim=-1) ** 2) + 1e-6
                h = F.relu(d_scale * (h_cent / torch.sqrt(var)))

            elif self.norm_type == "conical":
                w_norm = F.normalize(w, p=2, dim=0, eps=1e-7)
                h = torch.matmul(ax, w_norm)
                norm = torch.sqrt(torch.mean(h ** 2, dim=0, keepdim=True) + 1e-7)
                h = F.gelu(d_scale * (h / norm))

            elif self.norm_type == "equatorial":
                # Weight Standardization
                w_mean = w.mean(dim=0, keepdim=True)
                w_cent = w - w_mean
                w_norm = w_cent / (torch.norm(w_cent, p=2, dim=0, keepdim=True) + 1e-5)
                h = torch.matmul(ax, w_norm)

                # Project out constant eigenvector across nodes (zero mean)
                u = h.mean(dim=0, keepdim=True)
                h_cent = h - u
                # Geodesic retraction onto S^{N-2}
                norm = torch.sqrt(torch.mean(h_cent ** 2, dim=0, keepdim=True) + 1e-6)
                h = F.silu(d_scale * (h_cent / norm))

            else:
                raise ValueError(f"Unknown norm_type: {self.norm_type}")

        return h

    def forward(self, x: torch.Tensor, adj_hat: torch.Tensor) -> torch.Tensor:
        return self.head(self.extract_features(x, adj_hat))


# -----------------------------------------------------------------------------
# 4. Multi-Seed Benchmark Execution Loop
# -----------------------------------------------------------------------------
def run_benchmark() -> None:
    (
        x, adj_hat, y,
        train_mask, val_mask, test_mask,
        in_dim, num_classes
    ) = load_cora_dataset()

    models_to_test = [
        ("Vanilla GCN", "vanilla"),
        ("LayerNorm GCN", "layernorm"),
        ("PairNorm GCN", "pairnorm"),
        ("Graph-Conical", "conical"),
        ("Graph-Equatorial", "equatorial"),
    ]

    results = {
        m[0]: {d: {"acc": [], "pwd": [], "dirich": [], "dead": []} for d in DEPTHS}
        for m in models_to_test
    }

    print("=" * 120)
    print(f"[INFO] Starting GNN Over-Smoothing Benchmark on Cora | Device: {DEVICE}")
    print(f"[INFO] Seeds: {SEEDS} | Depths: {DEPTHS} | Epochs per Model: 150")
    print("=" * 120)

    start_time = time.time()

    for seed_idx, seed in enumerate(SEEDS, 1):
        print(f"\n[INFO] Seed [{seed_idx}/{len(SEEDS)}]: {seed}")
        print("-" * 120)

        for depth in DEPTHS:
            for name, norm_type in models_to_test:
                torch.manual_seed(seed)
                if torch.cuda.is_available():
                    torch.cuda.manual_seed_all(seed)

                model = DeepGraphModel(
                    in_dim=in_dim,
                    hidden_dim=64,
                    num_classes=num_classes,
                    num_layers=depth,
                    norm_type=norm_type,
                ).to(DEVICE)

                optimizer = torch.optim.AdamW(model.parameters(), lr=0.01, weight_decay=5e-4)
                criterion = nn.CrossEntropyLoss()

                model.train()
                for _ in range(150):
                    optimizer.zero_grad()
                    out = model(x, adj_hat)
                    loss = criterion(out[train_mask], y[train_mask])
                    loss.backward()
                    optimizer.step()

                model.eval()
                with torch.no_grad():
                    latents = model.extract_features(x, adj_hat)
                    preds = model.head(latents)
                    acc = float(
                        (preds[test_mask].argmax(dim=-1) == y[test_mask]).float().mean().item() * 100.0
                    )
                    pwd, dirich, _, dead = compute_graph_metrics_safe(latents, adj_hat)

                results[name][depth]["acc"].append(acc)
                results[name][depth]["pwd"].append(pwd)
                results[name][depth]["dirich"].append(dirich)
                results[name][depth]["dead"].append(dead)

                del model, optimizer
                torch.cuda.empty_cache()

            print(f"  [RUN] Completed depth L={depth:>2} for all architectures.")

    total_duration = time.time() - start_time
    print(f"\n[INFO] Benchmark completed in {total_duration:.1f}s")

    # -------------------------------------------------------------------------
    # 5. Tabulated Performance Reports
    # -------------------------------------------------------------------------
    print("\n" + "=" * 120)
    print("TEST ACCURACY (%) VS. PROPAGATION DEPTH (5 SEEDS: MEAN +/- STD)")
    print("=" * 120)
    header = f"{'ARCHITECTURE':<20} | " + " | ".join([f"L={d:<2}             " for d in DEPTHS])
    print(header)
    print("-" * 120)
    for name, _ in models_to_test:
        columns = []
        for d in DEPTHS:
            acc_vals = results[name][d]["acc"]
            columns.append(f"{np.mean(acc_vals):>5.1f}% +/- {np.std(acc_vals):<4.1f}")
        print(f"{name:<20} | " + " | ".join(columns))
    print("=" * 120)

    print("\n" + "=" * 120)
    print("CRITICAL OVER-SMOOTHING AUDIT AT DEPTH L=32 (5 SEEDS: MEAN +/- STD)")
    print("=" * 120)
    print(
        f"{'ARCHITECTURE':<20} | {'TEST ACC (%)':<16} | {'PAIRWISE DIST':<18} | "
        f"{'DIRICHLET ENERGY':<18} | {'DEAD NODES (%)'}"
    )
    print("-" * 120)
    for name, _ in models_to_test:
        acc_m, acc_s = np.mean(results[name][32]["acc"]), np.std(results[name][32]["acc"])
        pwd_m, pwd_s = np.mean(results[name][32]["pwd"]), np.std(results[name][32]["pwd"])
        dir_m, dir_s = np.mean(results[name][32]["dirich"]), np.std(results[name][32]["dirich"])
        dead_m = np.mean(results[name][32]["dead"])

        print(
            f"{name:<20} | {acc_m:>5.1f}% +/- {acc_s:<5.1f} | {pwd_m:>6.4f} +/- {pwd_s:<6.4f} | "
            f"{dir_m:>7.4f} +/- {dir_s:<6.4f} | {dead_m:>12.1f}%"
        )
    print("=" * 120)


if __name__ == "__main__":
    run_benchmark()

[INFO] Starting GNN Over-Smoothing Benchmark on Cora | Device: cuda
[INFO] Seeds: [42, 1337, 2026, 777, 999] | Depths: [2, 4, 8, 16, 32, 64] | Epochs per Model: 150

[INFO] Seed [1/5]: 42
------------------------------------------------------------------------------------------------------------------------
  [RUN] Completed depth L= 2 for all architectures.
  [RUN] Completed depth L= 4 for all architectures.
  [RUN] Completed depth L= 8 for all architectures.
  [RUN] Completed depth L=16 for all architectures.
  [RUN] Completed depth L=32 for all architectures.
  [RUN] Completed depth L=64 for all architectures.

[INFO] Seed [2/5]: 1337
------------------------------------------------------------------------------------------------------------------------
  [RUN] Completed depth L= 2 for all architectures.
  [RUN] Completed depth L= 4 for all architectures.
  [RUN] Completed depth L= 8 for all architectures.
  [RUN] Completed depth L=16 for all architectures.
  [RUN] Completed depth L